# 03. Grid Graph Problems

A grid is a graph where each cell has up to 4 neighbours. These are the **most common graph questions** in interviews.

**Learning order:** TEMPLATE → ISLANDS → MAX AREA → FLOOD FILL → SURROUNDED REGIONS → PACIFIC ATLANTIC → WALLS AND GATES → INTERVIEW THINKING

## 1. Template

```python
DIRS = [(1,0), (-1,0), (0,1), (0,-1)]
def dfs(r, c):
    if not (0 <= r < rows and 0 <= c < cols) or grid[r][c] != target:
        return
    grid[r][c] = MARK                    # visited (or use a set)
    for dr, dc in DIRS:
        dfs(r + dr, c + dc)
```
Marking the grid itself saves memory but **modifies the input**: mention it, or use a `visited` set.

In [1]:
from collections import deque

DIRS = [(1, 0), (-1, 0), (0, 1), (0, -1)]

def num_islands(grid):
    rows, cols, count = len(grid), len(grid[0]), 0
    def dfs(r, c):
        if not (0 <= r < rows and 0 <= c < cols) or grid[r][c] != "1":
            return
        grid[r][c] = "0"                               # sink the island
        for dr, dc in DIRS:
            dfs(r + dr, c + dc)
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == "1":
                dfs(r, c)
                count += 1
    return count

g = [["1", "1", "0", "0", "0"], ["1", "1", "0", "0", "0"], ["0", "0", "1", "0", "0"], ["0", "0", "0", "1", "1"]]
print(num_islands(g))   # 3

3


## 2. Max Area of Island

DFS returns the **size** of the island it explores: `1 + sum of the neighbours' sizes`.

In [2]:
def max_area_of_island(grid):
    rows, cols = len(grid), len(grid[0])
    def dfs(r, c):
        if not (0 <= r < rows and 0 <= c < cols) or grid[r][c] != 1:
            return 0
        grid[r][c] = 0
        return 1 + sum(dfs(r + dr, c + dc) for dr, dc in DIRS)
    return max((dfs(r, c) for r in range(rows) for c in range(cols)), default=0)

print(max_area_of_island([[1, 1, 0], [1, 0, 0], [0, 0, 1]]))   # 3

3


## 3. Flood Fill (paint bucket)

Change the starting cell's color and spread to connected cells with the **same original color**. If the new color equals the old one, return immediately (otherwise infinite loop).

In [3]:
def flood_fill(image, sr, sc, color):
    old = image[sr][sc]
    if old == color:
        return image
    rows, cols = len(image), len(image[0])
    def dfs(r, c):
        if 0 <= r < rows and 0 <= c < cols and image[r][c] == old:
            image[r][c] = color
            for dr, dc in DIRS:
                dfs(r + dr, c + dc)
    dfs(sr, sc)
    return image

print(flood_fill([[1, 1, 1], [1, 1, 0], [1, 0, 1]], 1, 1, 2))   # [[2, 2, 2], [2, 2, 0], [2, 0, 1]]

[[2, 2, 2], [2, 2, 0], [2, 0, 1]]


## 4. Surrounded Regions: think in reverse

Flip all `O`s that are **fully surrounded** by `X`. Hard to check directly. Reverse: an `O` is **safe** if it is connected to a border `O`. So:
1. Start DFS from every **border** `O`, mark them safe (`S`).
2. Flip all remaining `O` to `X`.
3. Turn `S` back into `O`.

**Reverse thinking** (start from the boundary or the target) appears in many grid problems.

In [4]:
def solve_surrounded(board):
    rows, cols = len(board), len(board[0])
    def dfs(r, c):
        if 0 <= r < rows and 0 <= c < cols and board[r][c] == "O":
            board[r][c] = "S"
            for dr, dc in DIRS:
                dfs(r + dr, c + dc)
    for r in range(rows):
        dfs(r, 0); dfs(r, cols - 1)
    for c in range(cols):
        dfs(0, c); dfs(rows - 1, c)
    for r in range(rows):
        for c in range(cols):
            board[r][c] = "O" if board[r][c] == "S" else "X"
    return board

b = [["X", "X", "X", "X"], ["X", "O", "O", "X"], ["X", "X", "O", "X"], ["X", "O", "X", "X"]]
print(solve_surrounded(b))   # [['X', 'X', 'X', 'X'], ['X', 'X', 'X', 'X'], ['X', 'X', 'X', 'X'], ['X', 'O', 'X', 'X']]

[['X', 'X', 'X', 'X'], ['X', 'X', 'X', 'X'], ['X', 'X', 'X', 'X'], ['X', 'O', 'X', 'X']]


## 5. Pacific Atlantic Water Flow

Water flows to **equal or lower** cells. Find cells that can reach **both** oceans. Reverse: start from the ocean borders and move **uphill** (to cells with height `>=`). A cell reachable from both searches is in the answer.

In [5]:
def pacific_atlantic(heights):
    rows, cols = len(heights), len(heights[0])
    def reach(starts):
        seen, q = set(starts), deque(starts)
        while q:
            r, c = q.popleft()
            for dr, dc in DIRS:
                nr, nc = r + dr, c + dc
                if (0 <= nr < rows and 0 <= nc < cols and (nr, nc) not in seen
                        and heights[nr][nc] >= heights[r][c]):          # water could flow back down
                    seen.add((nr, nc))
                    q.append((nr, nc))
        return seen
    pacific = reach([(r, 0) for r in range(rows)] + [(0, c) for c in range(cols)])
    atlantic = reach([(r, cols - 1) for r in range(rows)] + [(rows - 1, c) for c in range(cols)])
    return sorted(map(list, pacific & atlantic))

h = [[1, 2, 2, 3, 5], [3, 2, 3, 4, 4], [2, 4, 5, 3, 1], [6, 7, 1, 4, 5], [5, 1, 1, 2, 4]]
print(pacific_atlantic(h))   # [[0, 4], [1, 3], [1, 4], [2, 2], [3, 0], [3, 1], [4, 0]]

[[0, 4], [1, 3], [1, 4], [2, 2], [3, 0], [3, 1], [4, 0]]


## 6. Walls and Gates (multi-source BFS)

Fill each empty room with its distance to the **nearest gate**. Instead of BFS from every room, put **all gates** in the queue at the start and run **one** BFS. The first time a room is reached is its shortest distance.

In [6]:
def walls_and_gates(rooms):
    INF = 2147483647
    rows, cols = len(rooms), len(rooms[0])
    q = deque((r, c) for r in range(rows) for c in range(cols) if rooms[r][c] == 0)
    while q:
        r, c = q.popleft()
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and rooms[nr][nc] == INF:
                rooms[nr][nc] = rooms[r][c] + 1
                q.append((nr, nc))
    return rooms

INF = 2147483647
print(walls_and_gates([[INF, -1, 0, INF], [INF, INF, INF, -1], [INF, -1, INF, -1], [0, -1, INF, INF]]))
# [[3, -1, 0, 1], [2, 2, 1, -1], [1, -1, 2, -1], [0, -1, 3, 4]]

[[3, -1, 0, 1], [2, 2, 1, -1], [1, -1, 2, -1], [0, -1, 3, 4]]


## 7. Interview thinking

**Signals:** "islands", "connected cells", "regions", "paint/fill", "reachable from the border/ocean", "nearest X for every cell".

**Say:** "Each cell is a node connected to its 4 neighbours. DFS/BFS marks components in O(R·C). For 'nearest' questions I use multi-source BFS."

**Common mistakes:** infinite recursion (not marking visited, or flood fill with the same color), bounds check order (`0 <= r < rows` before indexing), recursion depth on large grids (use BFS/iterative), modifying the input without saying so, wrong direction in reverse searches (`>=` when going uphill).

## Quick Revision

- Grid = graph. 4 directions, bounds check first, mark visited.
- Islands: DFS from each unvisited land cell, count starts.
- Area: DFS returns size.
- Surrounded / ocean problems: **reverse**, start from the border.
- "Nearest" for all cells: multi-source BFS.